In [0]:
spark.sql("use CATALOG `databricks-pyspark` ")

In [0]:
spark.sql("use SCHEMA spark_ns ")

In [0]:
spark.sql("DROP TABLE IF EXISTS employee_bronze")

In [0]:
dbutils.fs.rm('/dbfs/tmp/autoloader_demo', True)

In [0]:
schema_path = "/tmp/autoloader_demo/schema"
checkpoint_path = "/tmp/autoloader_demo/checkpoint"
source_path = "/tmp/autoloader_demo/source_path"
destination_path = "/tmp/autoloader_demo/destination_path"

In [0]:
base_path = "/Volumes/databricks-pyspark/databricks-pyspark-schema/internal-v1/autoloader_demo"
source_path = base_path + "/input"
schema_location = base_path + "/schema"
checkpoint_path = base_path + "/checkpoint"


In [0]:
dbutils.fs.put(
    source_path + "/employee_01.csv",
    """emp_id,name,salary
1,John,10000
2,Mary,20000
3,Mike,30000
""",
    True
)

In [0]:
display(dbutils.fs.ls(source_path))

In [0]:
df_input = spark.readStream.format("cloudFiles") \
                     .option("cloudFiles.format","csv") \
                     .option("cloudFiles.schemaLocation",schema_location) \
                     .option("header","true") \
                     .option("inferSchema","true") \
                     .load(source_path) 
                     
                  

In [0]:
query = df_input.writeStream \
                .format("delta") \
                    .mode("append")\
                   
                         .option("checkpointLocation",checkpoint_path) \
                          .trigger(availableNow=True) \
                          .toTable("employee_bronze")


In [0]:
spark.table("employee_bronze").show()

In [0]:
dbutils.fs.put(
    source_path + "/employee_02.csv",
    """emp_id,name,salary
4,Sara,40000
5,Rin,50000
""",
    True
)